# Baseline Campaign - Graph Classification

Datasets: MUTAG, PROTEINS, ENZYMES, NCI1.

TUDataset's protocol (tudataset/tud_benchmark/auxiliarymethods/gnn_evaluation.py): unstratified shuffled 10-fold, a further 10% of each training fold held out for validation, five repetitions - 50 runs per cell. Reported as Macro-F1 and Micro-F1, with two standard deviations: across the five repetition means, and across all fifty runs. Every baseline reads out with the same global mean pool + linear head.

Hyperparameters are selected **inside each fold**, as TUDataset's inner grid does, so a cell is 50 sweeps rather than one. That is the expensive choice - check the printed fit count in cell 3B before launching, and use UNIT_START/UNIT_COUNT to partition.

Only MUTAG has more than one edge type (4); PROTEINS, ENZYMES and NCI1 have one. The heterogeneous models therefore see a single relation on those three and run at their homogeneous limit - `num_relation_keys` on each record makes that visible. TU node types are the one-hot node labels, i.e. the features, so a homogeneous model loses nothing structural there either.

Cells 3A and 3B take explicit model lists, so a single cell can be run in isolation. Everything is driven by `scripts/run_baseline_published.py` and `scripts/run_baseline_tuning.py`; nothing is reimplemented here.

## 0. Environment

In [1]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/H2S-NN_test')
else:
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/H2S-NN_test


Install in two stages so a library problem cannot take the whole campaign down.

**Core** (`requirements/core.txt`) is torch, PyG and what `src/` itself needs — mandatory, and it must succeed. On a fresh Colab runtime it replaces Colab's own torch, so the cell kills the runtime afterwards; **rerun from the first cell once it restarts**. On a machine that already has the pinned versions it is a no-op and execution continues.

**Optional groups** (`requirements/baselines-*.txt`) each back a subset of the baselines. They install one pip call at a time, and a group that fails only disables the models it backs — those cells are then dropped from the campaign before anything launches, never mid-run.

In [2]:
import importlib.metadata
import os
import signal
import subprocess
import sys

# Core only. The baseline-specific groups install in the next cell, independently.
CORE_REQUIREMENTS = PROJECT_DIR / 'requirements' / 'core.txt'
if not CORE_REQUIREMENTS.exists():
    raise FileNotFoundError(f'Missing pinned requirements: {CORE_REQUIREMENTS}')

try:
    installed_torch = importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError:
    installed_torch = ''

# core.txt carries its own --extra-index-url and -f lines, so this single command pulls
# the CUDA 12.4 torch build and the matching torch_scatter/torch_sparse wheels.
target_ready = installed_torch.startswith('2.6.') and '+cu124' in installed_torch
print('installed torch distribution:', installed_torch or 'not installed')

core = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '-r', str(CORE_REQUIREMENTS)],
    capture_output=True,
    text=True,
)
if core.returncode != 0:
    print(core.stdout[-3000:])
    print(core.stderr[-3000:])
    raise RuntimeError(f'Core install failed with exit code {core.returncode}.')
print('Core stack installed.')

if not target_ready:
    # Colab ships its own torch. Replacing it mid-session leaves the previously imported
    # compiled extensions loaded, so the runtime has to restart before anything else runs.
    print('Pinned Torch/CUDA stack installed. Restarting the runtime; rerun from the first cell.')
    if IN_COLAB:
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError('Restart Python, then rerun the notebook from the first cell.')

installed torch distribution: 2.6.0+cu124
Core stack installed.


In [3]:
import torch
import torch_geometric
import torch_scatter
import torch_sparse

if not torch.__version__.startswith('2.6.'):
    raise RuntimeError(f'Expected torch 2.6.*, found {torch.__version__}. Rerun the install cell.')
if torch.version.cuda != '12.4':
    raise RuntimeError(f'Expected the CUDA 12.4 Torch build, found CUDA {torch.version.cuda}.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('device:', DEVICE)
if DEVICE == 'cuda':
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name, '|', round(properties.total_memory / 1024**3, 1), 'GiB')


torch: 2.6.0+cu124
torch_geometric: 2.8.0.post1
device: cpu


In [4]:
# One pip call per optional group. A failure here is reported, not raised: the models it
# backs are dropped from the campaign further down, and everything else still runs.
import subprocess
import sys

from src.baselines.runtime.dependencies import REQUIREMENT_GROUPS, refresh


def install_group(group) -> tuple[bool, str, list[str]]:
    """Install one group, retrying without build isolation if that is the failure."""
    base = [sys.executable, '-m', 'pip', 'install', '--upgrade', '-r', str(group.path)]
    result = subprocess.run(base, capture_output=True, text=True)
    refresh()
    if group.is_installed():
        return True, 'installed', []
    # torch_householder ships only an sdist whose setup.py does
    # `from torch.utils.cpp_extension import ...`; pip's default build isolation hides the
    # already-installed torch from it. Retrying without isolation is the documented fix.
    retry = subprocess.run(base + ['--no-build-isolation'], capture_output=True, text=True)
    refresh()
    if group.is_installed():
        return True, 'installed (--no-build-isolation)', []
    stderr = retry.stderr or result.stderr
    tail = [line for line in stderr.splitlines() if line.strip()][-3:]
    return False, 'failed', tail


GROUP_STATUS: dict[str, bool] = {}
for group_name, requirement_group in REQUIREMENT_GROUPS.items():
    if requirement_group.is_installed():
        GROUP_STATUS[group_name] = True
        print(f'OK    {group_name:<11} already present')
        continue
    installed, detail, error_tail = install_group(requirement_group)
    GROUP_STATUS[group_name] = installed
    if installed:
        print(f'OK    {group_name:<11} {detail} - {requirement_group.summary}')
        continue
    print(f'FAIL  {group_name:<11} {requirement_group.summary}')
    print(f'      still missing: {", ".join(requirement_group.missing_modules())}')
    for line in error_tail:
        print(f'      {line[:160]}')

OK    dgl         already present
OK    sheaf       installed - Householder orthogonal maps used by the two sheaf baselines
OK    h2gb        installed - the H2GB graphgym stack behind H2G-former
OK    hetero2net  installed - Hetero^2Net's metapath-dropping and logging dependencies


In [5]:
import importlib.metadata
from src.baselines.registry import MODELS as MODEL_SPECS
from src.baselines.runtime import _dgl_compat  # noqa: F401  (import for side effect)
from src.baselines.runtime.dependencies import (
    REQUIREMENT_GROUPS,
    groups_for,
    installed_groups,
)

for group_name, present in sorted(installed_groups().items()):
    backed = [
        spec.name
        for spec in MODEL_SPECS
        if any(group_name in groups_for(spec.name, task) for task in spec.tasks)
    ]
    if present:
        print(f'OK    {group_name:<11} backs {", ".join(backed)}')
    else:
        print(f'MISS  {group_name:<11} disables {", ".join(backed)}')
        missing = ', '.join(REQUIREMENT_GROUPS[group_name].missing_modules())
        print(f'      missing: {missing}')

if installed_groups().get('dgl'):
    import dgl

    from src.baselines.runtime._dgl_compat import dgl_supports_cuda

    dgl_version_info = "unavailable (unknown)"
    try:
        dgl_version_info = dgl.__version__
    except AttributeError:
        try:
            dgl_version_info = importlib.metadata.version("dgl")
        except importlib.metadata.PackageNotFoundError:
            dgl_version_info = "unavailable (metadata not found)"
        except Exception as e:
            dgl_version_info = f"unavailable (error: {type(e).__name__})"

    print()
    print(f'dgl: {dgl_version_info} | CUDA support: {dgl_supports_cuda()}')
    if not dgl_supports_cuda():
        print('  DGL-backed baselines will run on CPU even though torch has a GPU.')

OK    dgl         backs gcn, gat, han, sehgnn, hinormer, pshgcn
OK    h2gb        backs h2gformer
OK    hetero2net  backs hetero2net
OK    sheaf       backs nsd, csnn



AttributeError: module 'dgl' has no attribute '__version__'

In [ ]:
# The vendored baselines are cloned at the repository root, not pip-installed.
# This reports which are present before a campaign spends hours discovering a
# missing one. DGL's GraphBolt shim is applied on import where it is needed.
from src.baselines.runtime import available_vendors

missing = [name for name, present in available_vendors().items() if not present]
for name, present in sorted(available_vendors().items()):
    print(('OK  ' if present else 'MISSING '), name)
if missing:
    raise RuntimeError(f'Clone these at the repository root first: {missing}')



In [ ]:
import wandb

wandb_key = None
if IN_COLAB:
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None

if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    wandb.login()


## 1. Campaign settings

In [ ]:
TASK = 'graph_classification'

MODELS = ['all']      # or e.g. ['pshgcn', 'h2sgnn']
DATASETS = ['all']    # or a subset

N_TRIALS = 30
NUM_RUNS = 5
BASE_SEED = 42

# Units are 50 (repetition, fold) pairs for this task. Partition a campaign across
# machines with these; never run the same index twice concurrently.
UNIT_START = 0
UNIT_COUNT = None

RESUME = False
OUTPUT_DIR = Path('experiments/baselines')
WANDB_PROJECT = 'h2snn_baseline'
WANDB_MODE = 'online'
WANDB_TAGS = ['baselines', TASK]


Drop any model whose requirement group did not install. This happens **before** the
launch cells, so a missing library costs those cells and nothing else — and it is
reported, never silent.

In [ ]:
from src.baselines.registry import models_for_task
from src.baselines.runtime.dependencies import models_missing_dependencies

requested = (
    [spec.name for spec in models_for_task(TASK)] if MODELS == ['all'] else list(MODELS)
)
blocked = models_missing_dependencies(tuple(requested), TASK)
MODELS = [name for name in requested if name not in blocked]

for name, reason in blocked.items():
    print(f'SKIP  {name}: {reason}')
print(f'{len(MODELS)} of {len(requested)} models runnable: ' + ', '.join(MODELS))
if not MODELS:
    raise RuntimeError(
        'No runnable models: every requested model is missing a requirement group. '
        'Fix the failures reported by the group-install cell before launching.'
    )

## 2. Cell matrix

Every (model, dataset) cell, marked `published` (inherits the authors' own settings), `tune` (needs a sweep), or the reason it is not runnable at all.

In [ ]:
from src.baselines.registry import DATASETS as TASK_DATASETS
from src.baselines.registry import (
    get_adapter,
    models_for_task,
    unsupported_reason,
)

rows = []
for spec in models_for_task(TASK):
    for dataset in TASK_DATASETS[TASK]:
        reason = unsupported_reason(spec.name, TASK, dataset)
        if reason:
            status = 'not runnable'
        elif get_adapter(spec.name).published_params(TASK, dataset) is not None:
            status = 'published'
        else:
            status = 'tune'
        rows.append((spec.display, dataset, status, reason or ''))

print(f"{'model':<16}{'dataset':<12}{'status':<14}reason")
for display, dataset, status, reason in rows:
    print(f'{display:<16}{dataset:<12}{status:<14}{reason[:70]}')

from collections import Counter
print()
print(Counter(status for _, _, status, _ in rows))


## 3A. Published-hyperparameter runs

No sweep: these cells inherit settings their authors published for that exact dataset. `--list` first to see which qualify.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, 'scripts/run_baseline_published.py', '--task', TASK, '--list'],
    check=True,
)


In [ ]:
cmd = [
    sys.executable, 'scripts/run_baseline_published.py',
    '--task', TASK,
    '--models', *MODELS,
    '--datasets', *DATASETS,
    '--output_dir', str(OUTPUT_DIR),
    '--num_runs', str(NUM_RUNS),
    '--base_seed', str(BASE_SEED),
    '--unit_start', str(UNIT_START),
]
if UNIT_COUNT is not None:
    cmd += ['--unit_count', str(UNIT_COUNT)]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 3B. Tuned runs

Bayes + Hyperband, one sweep per unit, one fresh worker process per trial - the same engine the H2SNN campaign uses. **Validate first**: the printed fit count is the whole cost of the launch.

In [ ]:
tune_cmd = [
    sys.executable, 'scripts/run_baseline_tuning.py',
    '--task', TASK,
    '--models', *MODELS,
    '--datasets', *DATASETS,
    '--output_dir', str(OUTPUT_DIR),
    '--n_trials', str(N_TRIALS),
    '--num_runs', str(NUM_RUNS),
    '--base_seed', str(BASE_SEED),
    '--unit_start', str(UNIT_START),
    '--wandb_project', WANDB_PROJECT,
    '--wandb_mode', WANDB_MODE,
    '--wandb_tags', *WANDB_TAGS,
]
if UNIT_COUNT is not None:
    tune_cmd += ['--unit_count', str(UNIT_COUNT)]
if RESUME:
    tune_cmd.append('--resume')

print('Validating:', ' '.join(tune_cmd + ['--validate_only']))
subprocess.run(tune_cmd + ['--validate_only'], check=True)


In [ ]:
# Launch. Re-run this cell with RESUME = True after a Colab restart: compatible
# per-unit states resume their original sweeps, and incompatible ones are
# preserved while numbered continuation sweeps are created.
print('Starting:', ' '.join(tune_cmd))
subprocess.run(tune_cmd, check=True)


## 4. Aggregate

Collects every finished cell into one CSV plus a LaTeX fragment in the manuscript's column order.

In [ ]:
import json

import pandas as pd

from src.utils.run_reports import report_metrics_for_task

metrics = report_metrics_for_task(TASK)
records = []
for summary_path in sorted(OUTPUT_DIR.glob('*/*_summary.json')):
    payload = json.loads(summary_path.read_text(encoding='utf-8'))
    summary = payload.get('summary', {})
    if summary.get('task') != TASK:
        continue
    model, _, dataset = summary_path.parent.name.partition(f'_{TASK}_')
    row = {
        'model': model,
        'dataset': dataset,
        'source': summary_path.stem.replace('_summary', ''),
        'units': summary.get('num_units', 0),
    }
    for metric in metrics:
        entry = summary.get('metrics', {}).get(metric, {})
        row[metric] = entry.get('mean')
        row[f'{metric}_std'] = entry.get('std')
        if 'std_over_repetitions' in entry:
            row[f'{metric}_std_reps'] = entry['std_over_repetitions']
            row[f'{metric}_std_runs'] = entry['std_over_runs']
    records.append(row)

frame = pd.DataFrame(records)
results_path = Path('results') / f'baselines_{TASK}.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
frame.to_csv(results_path, index=False)
print('wrote', results_path)
frame


In [ ]:
# LaTeX fragment. Values come straight from the summaries; nothing is filled in
# for a cell that has not run, so a gap stays visibly a gap.
if not frame.empty:
    headline = metrics[0]
    pivot = frame.pivot_table(index='model', columns='dataset', values=headline)
    print(pivot.to_latex(float_format='%.2f', na_rep='--'))
else:
    print('No summaries yet - run cell 3A or 3B first.')


## 5. Resume and partition

Per-unit state lives under `OUTPUT_DIR/<cell>/tuning_state/`. To split one campaign across machines, keep `NUM_RUNS` and `BASE_SEED` fixed and give each machine a disjoint `UNIT_START`/`UNIT_COUNT` range.

In [ ]:
state_root = OUTPUT_DIR / 'tuning_state'
for state_path in sorted(state_root.glob('*/tuning_state_*.json')):
    state = json.loads(state_path.read_text(encoding='utf-8'))
    print(
        f"{state_path.parent.name:<40}{state_path.stem:<28}"
        f"{len(state.get('completed_records', []))}/{state.get('target_trials')} "
        f"sweep {state.get('sweep_id')}"
    )
